In [ ]:
!unzip labqar_pipeline.zip -d /content/
%cd /content/labqar_pipeline

In [ ]:
!pip install -q -r requirements.txt


In [ ]:
!pip install google-genai

In [ ]:
!python step1_build_registry.py        # -> artifacts/step1_reference_registry.*, corrected labels
!python step2_build_corpus.py          # -> artifacts/step2_corpus.jsonl
!python step3b_embeddings.py           # -> artifacts/step3b_embeddings.npz (downloads the model, needs internet)
!python step3c_vector_store.py         # -> artifacts/chroma_db/ (persisted vector DB)

In [ ]:
!python step5_evaluate.py              # structured retrieval + classification accuracy
!python step5b_generate_eval_queries.py
!python step5c_evaluate_rag.py         # semantic vs lexical retrieval, generation groundedness

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = ""
!python step6_llm_explain.py
!python step6b_freetext_rag_qa.py

In [ ]:
!git config --global user.email "berraegcin@gmail.com"
!git config --global user.name "berraEgcin"

In [ ]:
!git branch

In [ ]:
from step7_hybrid_checker import HybridRangeChecker
from step4_range_checker_rag import RangeCheckerRAG
from step6_llm_explain import Explainer
from step6b_freetext_rag_qa import RagQA

def run_tests():
    print("Loading system components...\n")

    # 1. Init modules
    # Primary decision from CSV, audit via LabQAR
    hybrid_checker = HybridRangeChecker()
    rag_checker = RangeCheckerRAG() # For RAG explanation
    explainer = Explainer() # Fallback if no API key
    qa = RagQA()

    print("-" * 50)
    print("TEST 1: Structured Data Check")
    print("-" * 50)
    test_cases = [
        ("ALT", 52.0, "U/L", "Female"),
        ("Hemoglobin", 11.1, "g/dL", "Male")
    ]

    for param, val, unit, gender in test_cases:
        # Get primary decision from hybrid (Referans: Senin CSV'n)
        status = hybrid_checker.evaluate(param, val, gender=gender)
        print(f"Parameter: {param} | Value: {val} {unit} | Gender: {gender}")
        print(f"Karar (CSV'den): {status}")

        # Use LabQAR range for explanation
        rag_result = rag_checker.evaluate(param, val, gender=gender)
        summary = explainer.explain(param, val, unit, rag_result)
        print(f"LLM Explanation: {summary}\n")

    print("-" * 50)
    print("TEST 2: Serbest Metin RAG Soru-Cevap")
    print("-" * 50)
    soru = "what is a normal fasting glucose range for an adult?"
    # Lists documents if no API key
    cevap = qa.answer(soru, top_k=2)
    print(f"Soru: {soru}")
    print(f"Cevap:\n{cevap['answer']}\n")

    print("-" * 50)
    print("TEST 3: Denetim (Audit) Raporu")
    print("-" * 50)
    # Lists CSV vs LabQAR mismatches
    hybrid_checker.print_audit_report()

if __name__ == "__main__":
    run_tests()